# Stage B validation — §8.2 of the plan

Runs every block of §8.2 against the held-out window (Jan 2025 – Apr 2026 per the plan).

Prerequisites:
- B1 daily NetCDFs exist under `DAILY_DIR`.
- B2 kriged NetCDFs exist under `KRIGED_DIR`.
- B3 RF gap-filled NetCDFs exist under `GAPFILL_DIR` and a trained model under `MODELS_DIR/rf_primary.joblib`.

If any of those are missing, run `python run_stage_b.py all --start 2022-09-01 --end 2026-04-30` first.

In [ ]:
from datetime import date
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))   # so we can import stage_b modules
import validate as vb
import rf_gapfill as rf
import config as cfg

START = cfg.TEST_START
END   = cfg.TEST_END
print(f'Held-out window: {START} → {END}')

## §8.2.1 — Internal consistency for the RF candidate

Reads the metrics stored on the trained bundle and applies the ±15 % band around `rmse_cv_mean`.  The train and internal-test RMSEs must both fall inside the band; otherwise the candidate is rejected before AERONET-blind validation.

In [ ]:
bundle = rf.load_bundle()
print('Training window:', bundle.training_window)
print('Hyperparameters:', bundle.hyperparams)
print('Metrics       :', bundle.metrics)

diag = vb.internal_consistency(bundle.metrics)
pd.DataFrame([diag])

## §8.2.2 — AERONET-blind matched pairs (the headline test)

Keep only days where AERONET observed but the B1 daily product was missing at the AERONET cell.  Compare each candidate's filled value against AERONET.

In [ ]:
pairs_rf = vb.aeronet_pairs(START, END, candidate='rf', blind_only=True)
pairs_kr = vb.aeronet_pairs(START, END, candidate='kriging', blind_only=True)
print(f'AERONET-blind pairs — RF: {len(pairs_rf)}, kriging: {len(pairs_kr)}')
vb.metric_panel(pairs_rf)

In [ ]:
vb.metric_panel(pairs_kr)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for ax, (label, df) in zip(axes, [('RF gap-fill', pairs_rf), ('Kriging baseline', pairs_kr)]):
    if df.empty:
        ax.set_title(f'{label}: no matched pairs'); ax.set_visible(False); continue
    for site, marker in zip(df['site'].unique(), ('o', 's')):
        sub = df[df['site'] == site]
        ax.scatter(sub['aer_aod'], sub['sat_aod'], alpha=0.6, label=site, marker=marker)
    lim = max(df[['aer_aod', 'sat_aod']].max().max(), 1.0)
    ax.plot([0, lim], [0, lim], 'k--', lw=0.8)
    ax.set_xlabel('AERONET AOD'); ax.set_ylabel('Filled AOD'); ax.set_title(label)
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

## §8.2.3 — Coverage and `days_since_last_observed` audit

In [ ]:
cov = vb.coverage_audit(START, END)
cov

In [ ]:
if not cov.empty:
    ax = cov.plot(x='month', y=['pre_fill_observed', 'post_fill_coverage'],
                  kind='line', marker='o', figsize=(10, 4))
    ax.axhline(0.95, color='r', ls='--', lw=0.8, label='§9 target ≥95%')
    ax.set_ylim(0, 1.05); ax.set_ylabel('coverage fraction'); ax.legend(); ax.grid(alpha=0.3)

In [ ]:
vb.dso_stratified_rmse(pairs_rf)

## §8.2.4 — Robustness diagnostics

In [ ]:
vi = vb.variable_importance()
vi.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
vi.head(15).iloc[::-1].plot.barh(x='feature', y='pct_importance', ax=ax, legend=False)
ax.set_xlabel('Gini importance (%)'); ax.set_title('RF predictor importance')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()

In [ ]:
env = vb.residual_envelope(pairs_rf)
env

In [ ]:
if not env.empty:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.errorbar(env['aer_bin'], env['bias'], yerr=env['sigma'],
                fmt='o-', capsize=3, lw=1.2)
    ax.axhline(0, color='k', lw=0.6)
    ax.set_xlabel('AERONET AOD bin'); ax.set_ylabel('residual (sat − AERONET)')
    ax.set_title('Residual envelope vs AERONET AOD (Lee 2025 Fig. 7 analogue)')
    ax.grid(alpha=0.3)

## §8.2.5 — Cloud-period recovery check

In [ ]:
vb.cloud_period_recovery(START, END)

## §8.2.6 — Success-criteria table

The headline §9 table — baseline / target / achieved.  Achieved values come from the RF blind pairs and the coverage audit above.

In [ ]:
pairs_full = vb.aeronet_pairs(START, END, candidate='rf', blind_only=False)
vb.success_table(pairs_full, cov)